In [ ]:
# Copyright 2025 Sysco
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Lab 6.6: Google Cloud Data and Analytics Architecture for ML


## Module Learning Objectives

By the end of this lab, you will be able to:
- Understand how BigQuery datasets integrate with Vertex AI.
- Create Vertex AI dataset resources from BigQuery tables.
- Export BigQuery tables to Cloud Storage for AutoML and custom training.
- Load BigQuery data into pandas, TensorFlow, and XGBoost training pipelines.
- Apply best practices for structured data in end-to-end MLOps workflows.


### Sysco MLOps Framework Best Practices for Structured Data in BigQuery

When doing MLOps on Google Cloud, following are the best practices when dealing with structured (tabular) data in BigQuery:

- For AutoML training:
  - Create a managed dataset with Vertex AI TabularDataset.
  - Use the BigQuery table as the input to the dataset.
  - Specify columns and columns transformations when running the AutoML training pipeline job.


- For custom training:
  - For small datasets:
    - Extract the BigQuery to a pandas dataframe.
    - Preprocess the data in the dataframe.
  - For large datasets:
    - TensorFlow model training:
      - Create a tf.data.Dataset generator from the BigQuery table.
      - Specify the columns for the custrom training.
      - Preprocess the data either:
        - Within the generator (upstream)
        - Within the model (downstream)
    - XGBoost model training:
      - Use BigQuery ML built-in XGBoost training.
      - Alternatively, create a DMatrix generator from CSV files extracted from BigQuery table.
    - PyTorch model training:
        - Extract the BigQuery to a pandas dataframe.
        - Preprocess the data in the dataframe.
        - Create a DataLoader generator from the pandas dataframe.


- Alternatively:
    - Extract the BigQuery table to CSV files.
    - Preprocess the CSV files.
    - Create a tf.data.Dataset generator from the CSV files.

## Get started

### Install Vertex AI SDK for Python and other required packages


In [ ]:
# 1. Uninstall conflicting packages

!pip uninstall -y google-auth google-auth-oauthlib google-cloud-bigquery \ google-cloud-aiplatform google-cloud-bigquery-storage pyarrow db-dtypes \ tensorflow tensorflow-io xgboost numpy pandas

In [ ]:
# 2. Reinstall all required packages with compatible versions
!pip install --upgrade --quiet \ google-cloud-aiplatform \ google-cloud-bigquery \ google-cloud-bigquery-storage \ tensorflow \ tensorflow-io \ xgboost \ numpy \ pandas \ pyarrow \ db-dtypes \ google-auth==2.45.0 \ google-auth-oauthlib

In [ ]:
import IPython
IPython.Application.instance().kernel.do_shutdown(True)


In [ ]:
import sys

if "google.colab" in sys.modules:
    # Authenticate in Colab
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated with Colab user credentials.")
else:
    # For local or Vertex AI Workbench environments, use gcloud
    # Run this once in your terminal:
    #   gcloud auth application-default login
    print("Not running in Colab. Make sure you have run 'gcloud auth application-default login' locally or are using a service account in Vertex AI Workbench.")


### Set Google Cloud project information and initialize Vertex AI SDK for Python
- Safely ignore the future upgrade and deprecation warnings for this lab.
- To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com). Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


In [ ]:
# Set Google Cloud project information and initialize Vertex AI SDK for Python

# Replace with your actual project ID and region
PROJECT_ID = "mfav2-374520"  # @param {type:"string"}
LOCATION = "us-east1"  # @param {type:"string"}

from google.cloud import aiplatform

# Initialize Vertex AI SDK
aiplatform.init(project=PROJECT_ID, location=LOCATION)

print(f"Vertex AI initialized for project: {PROJECT_ID} in {LOCATION}")


### Create a Cloud Storage bucket

Create a storage bucket to store intermediate artifacts such as datasets.

In [ ]:
BUCKET_URI = f"gs://churn-TODO_replacewithyourname-{PROJECT_ID}-unique"  # @param {type:"string"}

**If your bucket doesn't already exist**: Run the following cell to create your Cloud Storage bucket.

In [ ]:
! gsutil mb -l $LOCATION $BUCKET_URI

### Import libraries and define constants

In [ ]:
# Import core libraries for data processing and ML

import pandas as pd          # Dataframes and preprocessing
import xgboost as xgb        # Gradient boosting ML library
from google.cloud import bigquery  # BigQuery client for data access


### Create BigQuery client

Create the BigQuery client.

In [ ]:
SERVICE_ACCOUNT = "vertex-pipeline-executor@mfav2-374520.iam.gserviceaccount.com"  # @param {type:"string"}

In [ ]:
import sys

IS_COLAB = "google.colab" in sys.modules

if (
    SERVICE_ACCOUNT == ""
    or SERVICE_ACCOUNT is None
    or SERVICE_ACCOUNT == "[your-service-account]"
):
    # Get your service account from gcloud
    if not IS_COLAB:
        shell_output = !gcloud auth list 2>/dev/null
        SERVICE_ACCOUNT = shell_output[2].replace("*", "").strip()

    if IS_COLAB:
        shell_output = ! gcloud projects describe $PROJECT_ID
        project_number = shell_output[-1].split(":")[1].strip().replace("'", "")
        SERVICE_ACCOUNT = f"{project_number}-compute@developer.gserviceaccount.com"

# Always print, regardless of condition
print("Service Account:", SERVICE_ACCOUNT)


In [ ]:
# Initialize BigQuery client

from google.cloud import bigquery

# If you authenticated with gcloud or Colab auth, this will work directly
bqclient = bigquery.Client(project=PROJECT_ID)

print("BigQuery client initialized for project:", PROJECT_ID)


#### Location of BigQuery training data.

Now, set the variable `IMPORT_FILE` to the location of the data table in BigQuery and `BQ_TABLE` with the table id.

In [ ]:
# Define the BigQuery source table

# IMPORT_FILE is used when creating a Vertex AI TabularDataset from BigQuery
IMPORT_FILE = "bq://bigquery-public-data.samples.gsod"

# BQ_TABLE is used when querying the table directly with BigQuery client
BQ_TABLE = "bigquery-public-data.samples.gsod"

print("BigQuery source set to:", BQ_TABLE)


### Create the dataset

#### BigQuery input data

Next, create the dataset resource using the `create` method for the `TabularDataset` class, which takes the following parameters:

- `display_name`: The human readable name for the dataset resource.
- `bq_source`: Import data items from a BigQuery table into the dataset resource.
- `labels`: User defined metadata. In this example, you store the location of the Cloud Storage bucket containing the user defined data.

Learn more about [TabularDataset from BigQuery table](https://cloud.google.com/vertex-ai/docs/datasets/create-dataset-api#aiplatform_create_dataset_tabular_bigquery_sample-python).

In [ ]:
# Use a unique display name so resources don't collide
dataset = aiplatform.TabularDataset.create(
    display_name="Use a unique display name",   # e.g., "noaa_weather_david"
    bq_source=[IMPORT_FILE],
    labels={"user_metadata": BUCKET_URI[5:]},
)


label_column = "mean_temp"

print(dataset.resource_name)

# Create the TabularDataset from BigQuery table takes time. Please be patient.

### Copy the dataset to Cloud Storage

Next, you make a copy of the BigQuery table as a CSV file, to Cloud Storage using the BigQuery extract command.

Learn more about [BigQuery command line interface](https://cloud.google.com/bigquery/docs/reference/bq-cli-reference).

In [ ]:
# Export BigQuery table to Cloud Storage as CSV

# Split the BQ_TABLE string into project, dataset, and table parts
comps = BQ_TABLE.split(".")
BQ_PROJECT_DATASET_TABLE = comps[0] + ":" + comps[1] + "." + comps[2]

# Use the bq CLI to export the table to Cloud Storage in CSV format
# BUCKET_URI must already exist and be unique per student
! bq --location=us extract --destination_format=CSV $BQ_PROJECT_DATASET_TABLE $BUCKET_URI/mydata*.csv

# List the exported CSV files in the bucket
IMPORT_FILES = ! gsutil ls $BUCKET_URI/mydata*.csv
print("Exported files:", IMPORT_FILES)

# Take the first file as an example
EXAMPLE_FILE = IMPORT_FILES[0]

# Preview the first few lines of the exported CSV
! gsutil cat $EXAMPLE_FILE | head


### Create the dataset

#### CSV input data

Next, create the dataset resource using the `create` method for the `TabularDataset` class, which takes the following parameters:

- `display_name`: The human readable name for the dataset resource.
- `gcs_source`: A list of one or more dataset index files to import the data items into the dataset resource.
- `labels`: User defined metadata. In this example, you store the location of the Cloud Storage bucket containing the user defined data.

Learn more about [TabularDataset from CSV files](https://cloud.google.com/vertex-ai/docs/datasets/create-dataset-api#aiplatform_create_dataset_tabular_gcs_sample-python)

In [ ]:
# Create a Vertex AI TabularDataset from CSV files in Cloud Storage

# gcs_source points to the list of CSV files exported from BigQuery
gcs_source = IMPORT_FILES

dataset = aiplatform.TabularDataset.create(
    display_name="enter_a_unique_name_here",   # e.g., "noaa_weather_david"
    gcs_source=gcs_source,                     # Cloud Storage CSV files
    labels={"user_metadata": BUCKET_URI[5:]},  # Optional metadata label
)

# Define the target column for ML tasks
label_column = "mean_temp"

# Print the dataset resource name (unique identifier in Vertex AI)
print("Dataset created:", dataset.resource_name)


### Create a view of the BigQuery dataset

Alternatively, you can create a logical view of a BigQuery dataset that has a subset of the fields.

Learn more about [Creating BigQuery views](https://cloud.google.com/bigquery/docs/views).

In [ ]:
# Set dataset name and view name in BigQuery
# MLOps best practice: use unique, descriptive names to avoid collisions and ensure traceability

BQ_MY_DATASET = "[your-dataset-name]"  # e.g., "mlops_dataset_david_20251218"
BQ_MY_TABLE = "[your-view-name]"       # e.g., "mlops_view_david_weather"

# Fallback to defaults if placeholders are not replaced
if (
    BQ_MY_DATASET == ""
    or BQ_MY_DATASET is None
    or BQ_MY_DATASET == "[your-dataset-name]"
):
    BQ_MY_DATASET = "mlops_dataset"

if BQ_MY_TABLE == "" or BQ_MY_TABLE is None or BQ_MY_TABLE == "[your-view-name]":
    BQ_MY_TABLE = "mlops_view"

print("Using dataset:", BQ_MY_DATASET)
print("Using view:", BQ_MY_TABLE)


In [ ]:
# Create the resources
! bq --location=US mk -d \
$PROJECT_ID:$BQ_MY_DATASET

sql_script = f'''
CREATE OR REPLACE VIEW `{PROJECT_ID}.{BQ_MY_DATASET}.{BQ_MY_TABLE}`
AS SELECT station_number,year,month,day,mean_temp FROM `{BQ_TABLE}`
'''
print(sql_script)

query = bqclient.query(sql_script)

### Read the BigQuery dataset into a pandas dataframe

Next, you read a sample of the dataset into a pandas dataframe using BigQuery `list_rows()` and `to_dataframe()` method, as follows:

- `list_rows()`: Performs a query on the specified table and returns a row iterator to the query results. Optionally specify:
 - `selected_fields`: Subset of fields (columns) to return.
 - `max_results`: The maximum number of rows to return. Same as SQL LIMIT command.


- `rows.to_dataframe()`: Invokes the row iterator and reads in the data into a pandas dataframe.

Learn more about [Loading BigQuery table into a dataframe](https://cloud.google.com/bigquery/docs/bigquery-storage-python-pandas)

In [ ]:
# Download the table.
table = bigquery.TableReference.from_string(BQ_TABLE)

rows = bqclient.list_rows(
    table,
    max_results=500,
    selected_fields=[
        bigquery.SchemaField("station_number", "STRING"),
        bigquery.SchemaField("year", "INTEGER"),
        bigquery.SchemaField("month", "INTEGER"),
        bigquery.SchemaField("day", "INTEGER"),
        bigquery.SchemaField("mean_temp", "FLOAT"),
    ],
)

dataframe = rows.to_dataframe()
print(dataframe.head())

# Clean up

To clean up all Google Cloud resources used in this project, you can [delete the Google Cloud
project](https://cloud.google.com/resource-manager/docs/creating-managing-projects#shutting_down_projects) you used for the tutorial.

Otherwise, you can delete the individual resources you created in this tutorial:

- Vertex AI dataset resource
- Cloud Storage Bucket
- BigQuery dataset

Set `delete_storage` to _True_ to delete the storage resources used in this notebook.

In [ ]:
import os

# Delete the dataset using the Vertex dataset object
dataset.delete()

# Delete the temporary BigQuery dataset
! bq rm -r -f $PROJECT_ID:$DATASET_ID

delete_storage = False
if delete_storage or os.getenv("IS_TESTING"):
    # Delete the created GCS bucket
    ! gsutil rm -r $BUCKET_URI
    # Delete the created BigQuery datasets
    ! bq rm -r -f $PROJECT_ID:$BQ_MY_DATASET